# Fork B.2 — Windowed 7-mer Graph + GNN (PyTorch only)

Pure PyTorch implementation (no PyG).

In [14]:
import os
from pathlib import Path
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    roc_auc_score, average_precision_score
)

def resolve_device(prefer: Optional[str] = None) -> torch.device:
    # prefer: None or 'cpu'/'mps'/'cuda'
    if prefer is not None:
        return torch.device(prefer)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

DEVICE = resolve_device()
print("Torch:", torch.__version__)
print("Device:", DEVICE)


Torch: 2.9.0
Device: mps


In [31]:
def read_fasta(path: Path) -> Tuple[List[str], List[str]]:
    headers, seqs = [], []
    cur = []
    head = None
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if head is not None:
                    seqs.append("".join(cur).upper())
                    cur = []
                head = line[1:].strip()
                headers.append(head)
            else:
                cur.append(line)
        if head is not None:
            seqs.append("".join(cur).upper())
    assert len(headers) == len(seqs)
    return headers, seqs

def load_feature_labels(path: Path) -> Dict[str, str]:
    """Load multiclass labels from features-tpase style file.
    Format: >header<tab>label (e.g., >hAT_1-aAnoBae#DNA/hAT\tDNA/hAT)
    Returns dict mapping header -> label string.
    """
    label_dict = {}
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split("\t")
            if len(parts) < 2:
                continue
            header = parts[0][1:] if parts[0].startswith(">") else parts[0]
            label_dict[header] = parts[1]
    return label_dict

def load_binary_feature_labels(path: Path) -> Dict[str, int]:
    """Load binary labels (1 if not None, 0 if None)."""
    label_dict = {}
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split("\t")
            if len(parts) < 2:
                continue
            header = parts[0][1:] if parts[0].startswith(">") else parts[0]
            label_dict[header] = 0 if parts[1] == "None" else 1
    return label_dict

def header_to_tag(header: str) -> str:
    """Extract TE class from header like 'hAT_1-aAnoBae#DNA/hAT' -> 'DNA/hAT'."""
    if "#" in header:
        return header.split("#")[1]  # Part after #
    return header

def build_multiclass_labels_from_headers(headers: List[str]):
    tags = [header_to_tag(h) for h in headers]
    uniq = sorted(set(tags))
    tag_to_id = {t:i for i,t in enumerate(uniq)}
    y = np.array([tag_to_id[t] for t in tags], dtype=np.int64)
    return y, tag_to_id, uniq

def make_binary_labels_from_tags(headers: List[str], negative_tag: str = "None") -> np.ndarray:
    tags = [header_to_tag(h) for h in headers]
    return np.array([0 if t == negative_tag else 1 for t in tags], dtype=np.int64)

In [16]:
# ASCII -> {0,1,2,3,4} for A,C,G,T,other
_ASCII_MAP = np.full(256, 4, dtype=np.uint8)
for ch,val in [("A",0),("C",1),("G",2),("T",3),("a",0),("c",1),("g",2),("t",3)]:
    _ASCII_MAP[ord(ch)] = val

_COMP = np.array([3,2,1,0], dtype=np.uint8)  # A<->T, C<->G in 0..3

def kmer_code_forward(arr4: np.ndarray) -> int:
    code = 0
    for v in arr4:
        code = (code << 2) | int(v)
    return code

def kmer_code_rc(arr4: np.ndarray) -> int:
    code = 0
    for v in arr4[::-1]:
        code = (code << 2) | int(_COMP[v])
    return code

def canonical_kmer_code(arr4: np.ndarray) -> int:
    c1 = kmer_code_forward(arr4)
    c2 = kmer_code_rc(arr4)
    return c1 if c1 < c2 else c2

def hash_u32(x: int, dim: int) -> int:
    z = (x * 0x9E3779B97F4A7C15) & 0xFFFFFFFFFFFFFFFF
    z ^= (z >> 33)
    z = (z * 0xC2B2AE3D27D4EB4F) & 0xFFFFFFFFFFFFFFFF
    z ^= (z >> 29)
    return int(z % dim)

@dataclass
class KmerWindowFeaturizer:
    k: int = 7
    dim: int = 2048
    window: int = 512
    stride: int = 256
    add_pos: bool = True
    l2_normalize: bool = True

    def featurize_sequence(self, seq: str):
        arr = _ASCII_MAP[np.frombuffer(seq.encode("ascii","ignore"), dtype=np.uint8)]
        L = int(arr.size)
        if L == 0:
            X = np.zeros((1, self.dim + (1 if self.add_pos else 0)), dtype=np.float32)
            return X, np.array([0], dtype=np.int64)

        if L <= self.window:
            starts = np.array([0], dtype=np.int64)
        else:
            starts = np.arange(0, L - self.window + 1, self.stride, dtype=np.int64)
            if starts.size == 0:
                starts = np.array([0], dtype=np.int64)

        out_dim = self.dim + (1 if self.add_pos else 0)
        X = np.zeros((starts.size, out_dim), dtype=np.float32)

        for wi, st in enumerate(starts):
            en = min(st + self.window, L)
            sub = arr[st:en]
            counts = np.zeros(self.dim, dtype=np.float32)
            total = 0

            k = self.k
            if sub.size >= k:
                for i in range(0, sub.size - k + 1):
                    kmer = sub[i:i+k]
                    if np.any(kmer == 4):
                        continue
                    code = canonical_kmer_code(kmer)
                    j = hash_u32(code, self.dim)
                    counts[j] += 1.0
                    total += 1

            if total > 0:
                counts /= float(total)

            if self.l2_normalize:
                nrm = np.linalg.norm(counts)
                if nrm > 0:
                    counts /= nrm

            if self.add_pos:
                center = (st + en) / 2.0
                pos = center / max(1.0, float(L))
                X[wi, :-1] = counts
                X[wi, -1] = pos
            else:
                X[wi, :] = counts

        return X, starts


In [17]:
@dataclass
class GraphSample:
    x: torch.Tensor
    edge_index: torch.Tensor
    y: torch.Tensor
    header: str

def build_chain_edge_index(n: int, undirected: bool = True, self_loops: bool = True) -> torch.Tensor:
    edges = []
    if n > 1:
        src = np.arange(n-1, dtype=np.int64)
        dst = np.arange(1, n, dtype=np.int64)
        edges.append((src, dst))
        if undirected:
            edges.append((dst, src))
    if self_loops:
        idx = np.arange(n, dtype=np.int64)
        edges.append((idx, idx))
    if not edges:
        ei = np.zeros((2,0), dtype=np.int64)
    else:
        s = np.concatenate([e[0] for e in edges])
        d = np.concatenate([e[1] for e in edges])
        ei = np.stack([s,d], axis=0)
    return torch.from_numpy(ei)

class WindowGraphDataset(torch.utils.data.Dataset):
    def __init__(self, headers: List[str], sequences: List[str], y: np.ndarray, featurizer: KmerWindowFeaturizer):
        self.headers=headers
        self.seqs=sequences
        self.y=y.astype(np.int64)
        self.feat=featurizer

    def __len__(self):
        return len(self.seqs)

    def __getitem__(self, idx: int) -> GraphSample:
        h=self.headers[idx]
        seq=self.seqs[idx]
        X,_=self.feat.featurize_sequence(seq)
        x=torch.from_numpy(X).to(torch.float32)
        ei=build_chain_edge_index(x.size(0), undirected=True, self_loops=True)
        y=torch.tensor(int(self.y[idx]), dtype=torch.int64)
        return GraphSample(x=x, edge_index=ei, y=y, header=h)

def collate_graphs(batch: List[GraphSample]):
    xs=[]; eis=[]; ys=[]; headers=[]
    batch_vecs=[]
    node_offset=0
    for gi,s in enumerate(batch):
        n=s.x.size(0)
        xs.append(s.x)
        eis.append(s.edge_index + node_offset)
        ys.append(s.y)
        headers.append(s.header)
        batch_vecs.append(torch.full((n,), gi, dtype=torch.int64))
        node_offset += n
    x=torch.cat(xs, dim=0)
    edge_index=torch.cat(eis, dim=1) if eis else torch.zeros((2,0),dtype=torch.int64)
    batch_vec=torch.cat(batch_vecs, dim=0)
    y=torch.stack(ys, dim=0)
    return x, edge_index, batch_vec, y, headers

def scatter_mean(x: torch.Tensor, idx: torch.Tensor, dim_size: int) -> torch.Tensor:
    out=torch.zeros((dim_size, x.size(1)), device=x.device, dtype=x.dtype)
    out.index_add_(0, idx, x)
    cnt=torch.bincount(idx, minlength=dim_size).clamp_min(1).to(x.device).to(x.dtype).unsqueeze(1)
    return out/cnt


In [18]:
class GraphSAGELayer(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, dropout: float = 0.1):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        src, dst = edge_index[0], edge_index[1]
        agg = torch.zeros_like(x)
        agg.index_add_(0, dst, x[src])
        deg = torch.bincount(dst, minlength=x.size(0)).clamp_min(1).to(x.device).to(x.dtype).unsqueeze(1)
        agg = agg / deg
        h = self.lin_self(x) + self.lin_neigh(agg)
        h = F.relu(h)
        return self.dropout(h)

class GNNClassifier(nn.Module):
    def __init__(self, in_dim: int, hidden: int, num_classes: int, n_layers: int = 3, dropout: float = 0.1):
        super().__init__()
        layers=[]
        d=in_dim
        for _ in range(n_layers):
            layers.append(GraphSAGELayer(d, hidden, dropout=dropout))
            d=hidden
        self.layers=nn.ModuleList(layers)
        self.head=nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden, num_classes),
        )

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor, batch_vec: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x, edge_index)
        B = int(batch_vec.max().item()) + 1 if batch_vec.numel() else 0
        g = scatter_mean(x, batch_vec, dim_size=B)
        return self.head(g)


In [32]:
def compute_class_weights(y: np.ndarray, num_classes: int) -> torch.Tensor:
    counts = np.bincount(y, minlength=num_classes).astype(np.float64)
    counts[counts == 0] = 1.0
    w = counts.sum() / counts
    w = w / w.mean()
    return torch.tensor(w, dtype=torch.float32)

class FocalLoss(nn.Module):
    """Focal Loss for handling class imbalance - focuses on hard examples."""
    def __init__(self, alpha: torch.Tensor = None, gamma: float = 2.0):
        super().__init__()
        self.alpha = alpha  # Class weights
        self.gamma = gamma  # Focusing parameter (0 = CE, higher = more focus on hard)
    
    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce_loss = F.cross_entropy(logits, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce_loss)  # Probability of correct class
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

def augment_reverse_complement(seq: str) -> str:
    """Return reverse complement of DNA sequence."""
    comp = {'A': 'T', 'T': 'A', 'C': 'G', 'G': 'C', 'N': 'N'}
    return ''.join(comp.get(c, 'N') for c in reversed(seq.upper()))

@torch.no_grad()
def predict_loader(model: nn.Module, loader, device: torch.device):
    model.eval()
    logits_list=[]; y_list=[]; headers=[]
    for x, edge_index, batch_vec, y, h in loader:
        x=x.to(device); edge_index=edge_index.to(device); batch_vec=batch_vec.to(device)
        logits=model(x, edge_index, batch_vec)
        logits_list.append(logits.detach().cpu())
        y_list.append(y.detach().cpu())
        headers.extend(h)
    logits=torch.cat(logits_list, dim=0) if logits_list else torch.empty((0,1))
    y=torch.cat(y_list, dim=0) if y_list else torch.empty((0,), dtype=torch.int64)
    return logits, y, headers

def evaluate_multiclass(logits: torch.Tensor, y: torch.Tensor) -> Dict[str, float]:
    y_np=y.numpy()
    probs=torch.softmax(logits, dim=1).numpy()
    pred=probs.argmax(axis=1)
    out={}
    out['acc']=accuracy_score(y_np, pred) if len(y_np) else float('nan')
    out['balanced_acc']=balanced_accuracy_score(y_np, pred) if len(y_np) else float('nan')
    try:
        out['auroc_macro_ovr']=float(roc_auc_score(y_np, probs, multi_class='ovr', average='macro'))
    except Exception:
        out['auroc_macro_ovr']=float('nan')
    try:
        Y=np.zeros_like(probs)
        Y[np.arange(len(y_np)), y_np]=1.0
        out['auprc_macro']=float(average_precision_score(Y, probs, average='macro'))
    except Exception:
        out['auprc_macro']=float('nan')
    return out

def run_train_gnn_kfold(
    fasta_path: Path,
    label_path: Path,
    exclude_tags: Optional[List[str]] = None,
    k: int = 7,
    dim: int = 2048,
    window: int = 512,
    stride: int = 256,
    hidden: int = 128,
    n_layers: int = 3,
    dropout: float = 0.1,
    batch_size: int = 16,
    epochs: int = 20,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    patience: int = 5,
    random_state: int = 42,
    min_class_count: Optional[int] = 20,
    device: Optional[torch.device] = None,
    num_workers: int = 0,
    # New parameters
    n_folds: int = 5,                    # K-fold CV on train/val
    test_size: float = 0.2,              # Held-out test set proportion
    use_focal_loss: bool = False,        # Use focal loss for imbalance
    focal_gamma: float = 2.0,            # Focal loss gamma
    oversample_minority: bool = False,   # Oversample minority classes
    use_reverse_complement: bool = False, # Data augmentation
):
    """
    Train GNN with K-fold cross-validation on train/val, fixed test set.
    All non-test data gets used for training across folds.
    """
    from sklearn.model_selection import StratifiedKFold
    
    device = resolve_device(None if device is None else str(device))
    print("Using device:", device)

    # Load and prepare data (same as before)
    headers, sequences = read_fasta(Path(fasta_path))
    print(f"Loaded {len(sequences)} sequences from FASTA")

    label_dict = load_feature_labels(Path(label_path))
    print(f"Loaded {len(label_dict)} labels from {label_path}")

    matched_headers, matched_seqs, matched_labels = [], [], []
    for h, s in zip(headers, sequences):
        if h in label_dict:
            matched_headers.append(h)
            matched_seqs.append(s)
            matched_labels.append(label_dict[h])
    
    headers, sequences, tags = matched_headers, matched_seqs, matched_labels

    if exclude_tags:
        exclude_set = set(exclude_tags)
        keep_mask = [t not in exclude_set for t in tags]
        excluded_count = sum(1 for m in keep_mask if not m)
        headers = [h for h, m in zip(headers, keep_mask) if m]
        sequences = [s for s, m in zip(sequences, keep_mask) if m]
        tags = [t for t, m in zip(tags, keep_mask) if m]
        print(f"Excluded {excluded_count} sequences, remaining: {len(sequences)}")

    # Build label mapping
    uniq = sorted(set(tags))
    tag_to_id = {t: i for i, t in enumerate(uniq)}
    id_to_tag = uniq
    y = np.array([tag_to_id[t] for t in tags], dtype=np.int64)

    # Drop rare classes
    if min_class_count and min_class_count > 1:
        counts = np.bincount(y, minlength=len(id_to_tag))
        print(f"\nClass distribution ({len(id_to_tag)} classes):")
        for i, cnt in sorted(enumerate(counts), key=lambda x: -x[1]):
            print(f"  {id_to_tag[i]}: {int(cnt)}")

        keep = np.where(counts >= min_class_count)[0]
        drop = np.where(counts < min_class_count)[0]

        if drop.size:
            keep_mask = np.isin(y, keep)
            dropped_n = int((~keep_mask).sum())
            print(f"\nDropping {dropped_n} sequences from {drop.size} classes with <{min_class_count} samples.")

            headers = [h for h, m in zip(headers, keep_mask) if m]
            sequences = [s for s, m in zip(sequences, keep_mask) if m]
            y = y[keep_mask]

            keep = keep.tolist()
            old_to_new = {int(old): new for new, old in enumerate(keep)}
            y = np.array([old_to_new[int(i)] for i in y], dtype=np.int64)
            id_to_tag = [id_to_tag[int(old)] for old in keep]

    num_classes = len(id_to_tag)
    print(f"\nFinal: {len(sequences)} sequences, {num_classes} classes")

    # --- Split: Fixed test set, K-fold on train ---
    idx_all = np.arange(len(sequences))
    idx_trainval, idx_te = train_test_split(
        idx_all, test_size=test_size, random_state=random_state, stratify=y
    )
    y_trainval = y[idx_trainval]
    
    print(f"Test set: {len(idx_te)} samples")
    print(f"Train+Val set: {len(idx_trainval)} samples ({n_folds}-fold CV)")

    # Featurizer
    feat = KmerWindowFeaturizer(k=k, dim=dim, window=window, stride=stride, add_pos=True, l2_normalize=True)
    in_dim = dim + 1

    # --- K-Fold Cross-Validation ---
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    fold_results = []
    all_val_logits, all_val_y, all_val_headers = [], [], []

    for fold_i, (train_idx_rel, val_idx_rel) in enumerate(skf.split(idx_trainval, y_trainval)):
        print(f"\n{'='*50}")
        print(f"FOLD {fold_i + 1}/{n_folds}")
        print(f"{'='*50}")
        
        # Map relative indices back to absolute
        idx_tr = idx_trainval[train_idx_rel]
        idx_va = idx_trainval[val_idx_rel]
        
        # Get data for this fold
        h_tr = [headers[i] for i in idx_tr]
        s_tr = [sequences[i] for i in idx_tr]
        y_tr = y[idx_tr]
        
        h_va = [headers[i] for i in idx_va]
        s_va = [sequences[i] for i in idx_va]
        y_va = y[idx_va]
        
        # Optional: Oversample minority classes
        if oversample_minority:
            counts = np.bincount(y_tr, minlength=num_classes)
            max_count = counts.max()
            new_h, new_s, new_y = list(h_tr), list(s_tr), list(y_tr)
            for cls in range(num_classes):
                cls_idx = np.where(y_tr == cls)[0]
                if len(cls_idx) < max_count:
                    n_add = max_count - len(cls_idx)
                    add_idx = np.random.choice(cls_idx, size=n_add, replace=True)
                    for ai in add_idx:
                        new_h.append(h_tr[ai])
                        new_s.append(s_tr[ai])
                        new_y.append(cls)
            h_tr, s_tr, y_tr = new_h, new_s, np.array(new_y, dtype=np.int64)
            print(f"  Oversampled: {len(y_tr)} training samples")
        
        # Optional: Add reverse complements
        if use_reverse_complement:
            aug_h, aug_s, aug_y = [], [], []
            for h, s, yi in zip(h_tr, s_tr, y_tr):
                aug_h.append(h + "_rc")
                aug_s.append(augment_reverse_complement(s))
                aug_y.append(yi)
            h_tr = list(h_tr) + aug_h
            s_tr = list(s_tr) + aug_s
            y_tr = np.concatenate([y_tr, np.array(aug_y, dtype=np.int64)])
            print(f"  With RC augmentation: {len(y_tr)} training samples")
        
        # Create datasets
        ds_tr = WindowGraphDataset(h_tr, s_tr, y_tr, feat)
        ds_va = WindowGraphDataset(h_va, s_va, y_va, feat)
        
        loader_tr = torch.utils.data.DataLoader(ds_tr, batch_size=batch_size, shuffle=True, num_workers=num_workers, collate_fn=collate_graphs)
        loader_va = torch.utils.data.DataLoader(ds_va, batch_size=batch_size, shuffle=False, num_workers=num_workers, collate_fn=collate_graphs)
        
        # Model
        model = GNNClassifier(in_dim=in_dim, hidden=hidden, num_classes=num_classes, n_layers=n_layers, dropout=dropout).to(device)
        class_w = compute_class_weights(y_tr, num_classes).to(device)
        
        if use_focal_loss:
            loss_fn = FocalLoss(alpha=class_w, gamma=focal_gamma)
        else:
            loss_fn = nn.CrossEntropyLoss(weight=class_w)
        
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        
        best_state = None
        best_score = -float("inf")
        bad = 0
        
        for ep in range(1, epochs + 1):
            # Train
            model.train()
            running = 0.0
            n_graphs = 0
            for x, edge_index, batch_vec, yb, _ in loader_tr:
                x, edge_index, batch_vec, yb = x.to(device), edge_index.to(device), batch_vec.to(device), yb.to(device)
                logits = model(x, edge_index, batch_vec)
                loss = loss_fn(logits, yb)
                opt.zero_grad(set_to_none=True)
                loss.backward()
                opt.step()
                running += float(loss.item()) * yb.size(0)
                n_graphs += yb.size(0)
            train_loss = running / max(1, n_graphs)
            
            # Validate
            model.eval()
            val_logits, val_y, _ = predict_loader(model, loader_va, device)
            m = evaluate_multiclass(val_logits, val_y)
            
            print(f"  Ep {ep:02d} | loss={train_loss:.4f} | bacc={m['balanced_acc']:.4f} | auroc={m['auroc_macro_ovr']:.4f}")
            
            score = m["balanced_acc"]
            if score > best_score + 1e-4:
                best_score = score
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                bad = 0
            else:
                bad += 1
                if bad >= patience:
                    print(f"  Early stopping at epoch {ep}")
                    break
        
        # Load best model and get validation predictions
        if best_state:
            model.load_state_dict(best_state)
            model.to(device)
        
        val_logits, val_y, val_headers = predict_loader(model, loader_va, device)
        fold_metrics = evaluate_multiclass(val_logits, val_y)
        fold_results.append(fold_metrics)
        
        all_val_logits.append(val_logits)
        all_val_y.append(val_y)
        all_val_headers.extend(val_headers)
        
        print(f"  Fold {fold_i+1} val: bacc={fold_metrics['balanced_acc']:.4f}, auroc={fold_metrics['auroc_macro_ovr']:.4f}")

    # --- Aggregate CV results ---
    print(f"\n{'='*50}")
    print("CROSS-VALIDATION SUMMARY")
    print(f"{'='*50}")
    
    cv_metrics = {}
    for key in fold_results[0].keys():
        values = [f[key] for f in fold_results]
        cv_metrics[key] = {"mean": np.mean(values), "std": np.std(values)}
        print(f"{key}: {cv_metrics[key]['mean']:.4f} ± {cv_metrics[key]['std']:.4f}")

    # --- Final: Train on ALL train+val, evaluate on test ---
    print(f"\n{'='*50}")
    print("FINAL MODEL (train on all train+val)")
    print(f"{'='*50}")
    
    h_trainval = [headers[i] for i in idx_trainval]
    s_trainval = [sequences[i] for i in idx_trainval]
    y_trainval_arr = y[idx_trainval]
    
    h_te = [headers[i] for i in idx_te]
    s_te = [sequences[i] for i in idx_te]
    y_te_arr = y[idx_te]
    
    ds_trainval = WindowGraphDataset(h_trainval, s_trainval, y_trainval_arr, feat)
    ds_te = WindowGraphDataset(h_te, s_te, y_te_arr, feat)
    
    loader_trainval = torch.utils.data.DataLoader(ds_trainval, batch_size=batch_size, shuffle=True, num_workers=num_workers, collate_fn=collate_graphs)
    loader_te = torch.utils.data.DataLoader(ds_te, batch_size=batch_size, shuffle=False, num_workers=num_workers, collate_fn=collate_graphs)
    
    model = GNNClassifier(in_dim=in_dim, hidden=hidden, num_classes=num_classes, n_layers=n_layers, dropout=dropout).to(device)
    class_w = compute_class_weights(y_trainval_arr, num_classes).to(device)
    loss_fn = FocalLoss(alpha=class_w, gamma=focal_gamma) if use_focal_loss else nn.CrossEntropyLoss(weight=class_w)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    # Train for fixed epochs (no early stopping - use CV to estimate best epoch)
    best_epoch = int(np.mean([len(f) for f in [fold_results]]))  # Rough estimate
    for ep in range(1, epochs + 1):
        model.train()
        for x, edge_index, batch_vec, yb, _ in loader_trainval:
            x, edge_index, batch_vec, yb = x.to(device), edge_index.to(device), batch_vec.to(device), yb.to(device)
            logits = model(x, edge_index, batch_vec)
            loss = loss_fn(logits, yb)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
        if ep % 2 == 0:
            print(f"  Epoch {ep}/{epochs}")
    
    # Test evaluation
    test_logits, test_y, test_headers = predict_loader(model, loader_te, device)
    test_metrics = evaluate_multiclass(test_logits, test_y)
    
    print(f"\nTEST RESULTS:")
    for k, v in test_metrics.items():
        print(f"  {k}: {v:.4f}")

    return {
        "model": model,
        "featurizer": feat,
        "cv_metrics": cv_metrics,
        "fold_results": fold_results,
        "test": {"logits": test_logits, "y": test_y, "headers": test_headers, "metrics": test_metrics},
        "id_to_tag": id_to_tag,
        "device": str(device),
    }

In [20]:
@torch.no_grad()
def predict_sequence(model: nn.Module, featurizer: KmerWindowFeaturizer, seq: str, device: torch.device):
    X,_ = featurizer.featurize_sequence(seq)
    x = torch.from_numpy(X).to(device)
    edge_index = build_chain_edge_index(x.size(0), undirected=True, self_loops=True).to(device)
    batch_vec = torch.zeros((x.size(0),), dtype=torch.int64, device=device)
    logits = model(x, edge_index, batch_vec).squeeze(0)
    return torch.softmax(logits, dim=0).detach().cpu().numpy()

def topk_classes(probs: np.ndarray, id_to_tag=None, k: int = 5):
    idx = np.argsort(-probs)[:k]
    return [((id_to_tag[i] if id_to_tag else int(i)), float(probs[i])) for i in idx]

def find_misclassified(logits: torch.Tensor, y: torch.Tensor, headers: List[str], id_to_tag=None, max_rows: int = 50):
    probs = torch.softmax(logits, dim=1).numpy()
    pred = probs.argmax(axis=1)
    y_np = y.numpy()
    bad = np.where(pred != y_np)[0]
    rows=[]
    for i in bad[:max_rows]:
        rows.append({"header":headers[i], "true": (id_to_tag[y_np[i]] if id_to_tag else int(y_np[i])), "pred": (id_to_tag[pred[i]] if id_to_tag else int(pred[i])), "p_pred": float(probs[i, pred[i]])})
    return rows, int(bad.size)

def save_checkpoint(path: Path, model: nn.Module, featurizer: KmerWindowFeaturizer, id_to_tag=None):
    ckpt={"model_state": model.state_dict(), "featurizer": featurizer.__dict__, "id_to_tag": id_to_tag}
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(ckpt, path)
    print("Saved:", path)

def load_checkpoint(path: Path, hidden: int, n_layers: int, dropout: float = 0.1, device: Optional[torch.device] = None):
    device = resolve_device(None if device is None else str(device))
    ckpt=torch.load(path, map_location="cpu")
    feat=KmerWindowFeaturizer(**ckpt["featurizer"])
    id_to_tag=ckpt.get("id_to_tag", None)
    state=ckpt["model_state"]
    num_classes = state["head.3.weight"].shape[0]
    in_dim = feat.dim + (1 if feat.add_pos else 0)
    model = GNNClassifier(in_dim=in_dim, hidden=hidden, num_classes=num_classes, n_layers=n_layers, dropout=dropout)
    model.load_state_dict(state, strict=True)
    model.to(device).eval()
    return model, feat, id_to_tag, device


## Example usage (edit paths)

```python
FASTA_PATH = Path("../data/vgp/all_vgp_tes.fa")

results = run_train_gnn(
    fasta_path=FASTA_PATH,
    label_mode="multiclass",
    subset_size=20000,
    batch_size=16,
    epochs=25,
    patience=5,
    dim=2048,
    window=512,
    stride=256,
)
```


In [ ]:
# --- Run experiment with K-fold CV ---
from pathlib import Path

FASTA_PATH = Path("../data/vgp/all_vgp_tes.fa")
LABEL_PATH = Path("../data/vgp/features-tpase")

if not FASTA_PATH.exists():
    raise FileNotFoundError(f"FASTA not found: {FASTA_PATH.resolve()}")
if not LABEL_PATH.exists():
    raise FileNotFoundError(f"Label file not found: {LABEL_PATH.resolve()}")

results = run_train_gnn_kfold(
    fasta_path=FASTA_PATH,
    label_path=LABEL_PATH,
    exclude_tags=["None"],            # Focus on labeled transposase sequences
    min_class_count=50,               # Increase for more reliable per-class metrics

    # K-fold and test split
    n_folds=5,                        # 5-fold CV on train+val
    test_size=0.2,                    # 20% held-out test

    # Class imbalance handling
    use_focal_loss=True,              # Focal loss (focuses on hard examples)
    focal_gamma=2.0,
    oversample_minority=False,        # Try True for more aggressive balancing
    use_reverse_complement=False,     # DNA augmentation

    # k-mer graph construction
    k=7,
    dim=2048,
    window=256,
    stride=256,

    # model/training - FAST TESTING
    hidden=128,
    n_layers=3,
    dropout=0.2,
    lr=1e-3,
    batch_size=32,
    epochs=5,
    patience=2,
    num_workers=0,

    random_state=42,
)

print("\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)
print(f"Classes: {len(results['id_to_tag'])}")
print(f"Class names: {results['id_to_tag']}")
print(f"\nCV Balanced Acc: {results['cv_metrics']['balanced_acc']['mean']:.3f} ± {results['cv_metrics']['balanced_acc']['std']:.3f}")
print(f"CV AUROC: {results['cv_metrics']['auroc_macro_ovr']['mean']:.3f} ± {results['cv_metrics']['auroc_macro_ovr']['std']:.3f}")
print(f"\nTest Balanced Acc: {results['test']['metrics']['balanced_acc']:.3f}")
print(f"Test AUROC: {results['test']['metrics']['auroc_macro_ovr']:.3f}")

Using device: mps
Loaded 135751 sequences from FASTA
Loaded 135751 labels from ../data/vgp/features-tpase
Matched 135751 sequences with labels
Excluded 125101 sequences with tags in ['None']
Remaining: 10650 sequences

Class distribution (17 classes):
  DNA/hAT: 4900
  DNA/TcMar-Tc1: 1875
  DNA: 1530
  DNA/PIF-Harbinger: 790
  DNA/PiggyBac: 563
  DNA/Academ-1: 415
  DNA/CMC: 180
  DNA/Sola-2: 93
  DNA/Kolobok: 86
  DNA/P: 77
  DNA/Sola-1: 43
  DNA/PIF-ISL2EU: 38
  DNA/MULE-MuDR: 23
  DNA/Crypton-V: 19
  DNA/Merlin: 12
  DNA/Ginger-1: 4
  DNA/Dada: 2

Dropping 37 sequences from 4 classes with <20 samples.

Final: 10613 sequences, 13 classes
Epoch 01 | train=2.4020 | val=2.2807 | acc=0.4075 | bacc=0.1391 | auroc=0.8192 | auprc=0.2116
Epoch 02 | train=2.0025 | val=1.7780 | acc=0.6095 | bacc=0.3173 | auroc=0.9175 | auprc=0.3520
Epoch 03 | train=1.6245 | val=1.5464 | acc=0.6861 | bacc=0.4472 | auroc=0.9411 | auprc=0.4256
Epoch 04 | train=1.3317 | val=1.5795 | acc=0.6767 | bacc=0.4485 | auro

In [33]:
# --- Alternative: Use FASTA header labels (all TE classes) ---
# This uses the full ~135k sequences with labels from FASTA headers
# The 8% transposase classes will be mixed with LTR, LINE, etc.

def run_train_gnn_fasta_labels(
    fasta_path: Path,
    exclude_tags: Optional[List[str]] = None,
    min_class_count: int = 50,
    **kwargs
):
    """Train using FASTA header labels instead of features-tpase file."""
    headers, sequences = read_fasta(Path(fasta_path))
    print(f"Loaded {len(sequences)} sequences")
    
    # Extract labels from FASTA headers
    tags = [header_to_tag(h) for h in headers]
    
    # Show distribution
    from collections import Counter
    tag_counts = Counter(tags)
    print(f"\nAll classes ({len(tag_counts)}):")
    for tag, cnt in tag_counts.most_common(20):
        print(f"  {tag}: {cnt}")
    if len(tag_counts) > 20:
        print(f"  ... and {len(tag_counts) - 20} more classes")
    
    # Count how many would remain after filtering
    kept = sum(1 for t, c in tag_counts.items() if c >= min_class_count and t not in (exclude_tags or []))
    print(f"\nWith min_class_count={min_class_count}, exclude={exclude_tags}:")
    print(f"  Would keep {kept} classes")
    
    return tag_counts

# Quick analysis: What happens if we use all FASTA data?
print("="*60)
print("ANALYSIS: Using FASTA header labels (all TE classes)")
print("="*60)
tag_counts = run_train_gnn_fasta_labels(
    fasta_path=Path("../data/vgp/all_vgp_tes.fa"),
    exclude_tags=["Unknown"],
    min_class_count=50,
)

ANALYSIS: Using FASTA header labels (all TE classes)
Loaded 135751 sequences

All classes (48):
  Unknown: 43919
  LTR/Gypsy: 15103
  LINE/L1: 12401
  DNA: 12094
  DNA/hAT: 11308
  LTR/DIRS: 9970
  LINE/CR1: 4242
  LINE/L2: 3666
  LTR/ERV1: 3161
  LTR: 2789
  DNA/TcMar-Tc1: 2283
  LTR/ERV2: 2202
  LTR/Pao: 1842
  LINE/RTE: 1683
  LINE/Rex-Babar: 1634
  DNA/PIF-Harbinger: 992
  DNA/PiggyBac: 937
  LTR/ERV3: 888
  LTR/Copia: 745
  DNA/Academ-1: 663
  ... and 28 more classes

With min_class_count=50, exclude=['Unknown']:
  Would keep 33 classes
